# Klasifikasi video berbasis audio: Kasar vs Nonkasar

Dataset: `D:\web\cnn_clasification\archive\Dataset`. Label diambil hanya dari folder utama **Kasar (0)** dan **Nonkasar (1)**. Subfolder seperti nama kata tidak menjadi kelas baru. Semua video berformat yang didukung dalam subfolder dipindai secara rekursif; JPEG, file non-video, dan salinan dalam `.ipynb_checkpoints` diabaikan. `SAMPLES_PER_CLASS = None` memakai seluruh video yang berhasil diproses.

## Audio penuh dari setiap video

FFmpeg mengekstrak **seluruh track audio pertama** setiap video, mengubahnya menjadi mono 22.050 Hz, tanpa batas dua detik dan tanpa memotong silence. Tidak ada ekstraksi frame gambar dan tidak perlu konversi manual menjadi WAV. Satu video menghasilkan satu sampel asli dan satu prediksi. Video tanpa audio, rusak, atau melewati timeout decoding dicatat di `skipped_files.json`; periksa log dan jumlah sampel sebelum melaporkan hasil. Timeout 600 detik adalah waktu proses, bukan batas durasi rekaman.

SVM-RBF dan DNN memakai 183 fitur statistik sepanjang audio. CNN memakai mel-spectrogram seluruh audio yang diringkas menjadi 64 x 256 dengan area averaging (interpolasi untuk rekaman pendek). Seluruh frame berkontribusi, namun peringkasan mengurangi resolusi waktu pada video panjang. Rekaman sangat pendek diberi padding minimum untuk perhitungan spektral.

## Evaluasi dan optimasi

Holdout 30% video asli disisihkan sebelum cross-validation pada development set. Setiap training partition memiliki validasi internal 20%. Ketiga model menggunakan indeks file yang sama; scaler hanya di-fit pada training dan augmentasi hanya digunakan untuk anggota training. Prediksi dan metrik dihitung per video asli. Accuracy, precision macro, recall macro, F1 macro, serta laporan per kelas dan confusion matrix disimpan.

Cache menyimpan fitur deterministik per isi video dan konfigurasi agar decoding/ekstraksi tidak diulang antar-fold. Augmentasi dihitung bertahap; seed tetap per file. CNN menggunakan GlobalAveragePooling2D untuk mengurangi parameter. Cache tidak berisi scaler atau hasil pemilihan model. Duplikat byte-identik dicatat dan disertakan sekali; duplikat dengan encoding berbeda serta video dari sumber/penutur yang sama tetap perlu diperiksa dan dikelompokkan sebelum klaim generalisasi ke sumber baru.

## Menjalankan

Pastikan kernel memiliki numpy, librosa, matplotlib, seaborn, scikit-learn, joblib, tensorflow, dan opsional tqdm. FFmpeg harus tersedia di PATH atau atur `FFMPEG_EXE` (fallback Windows: `C:\ffmpeg\bin\ffmpeg.exe`). Jalankan semua sel. Pelatihan semua video dengan augmentasi dan lima fold membutuhkan waktu, RAM, dan ruang disk lebih besar; peningkatan akurasi tidak dijamin.

Hasil tersimpan di `reviewer_results_video_binary/`; cache di `.audio_feature_cache_video/`. `video_inventory.csv` mencatat jumlah video ditemukan, `dataset_manifest.csv` mencatat video yang dipakai beserta label, durasi audio, dan SHA-256, sedangkan manifest split dan prediksi menunjukkan pembagian/evaluasi setiap file. Naikkan `CACHE_SCHEMA` jika mengubah definisi fitur/augmentasi. Gunakan direktori output baru untuk konfigurasi eksperimen berbeda.

Definisi 183 fitur dipertahankan: MFCC (66), delta/delta-delta (52), spektral (6), chroma (24), contrast (14), tonnetz (12), tempo (1), ZCR (2), RMS (2), dan mel (4). MFCC 13 dan 20 memiliki informasi yang tumpang tindih; jelaskan dalam metode.


In [1]:
# -*- coding: utf-8 -*-
"""Reviewer revision: shared-split SVM-RBF, DNN and CNN evaluation.
Original recordings are split before training-only augmentation.
All model selection uses internal validation, never the test partition.
See the notebook introduction for configuration and reporting limitations.
"""
import os
import sys
import traceback
import numpy as np
import librosa
from pathlib import Path
import warnings
# Warnings remain visible for experiment auditing.

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.svm import SVC
import csv
import json
import platform
from importlib.metadata import version
import joblib
import hashlib
import shutil
import subprocess

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, Conv2D, MaxPooling2D, GlobalAveragePooling2D, BatchNormalization, Activation
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Optional progress bar
try:
    from tqdm import tqdm
    TQDM_AVAILABLE = True
except Exception:
    TQDM_AVAILABLE = False

# ---------- Konfigurasi ----------
DATA_DIR = r"D:\web\cnn_clasification\archive\Dataset"  # Ganti sesuai dataset Anda
CLASS_NAMES = ['Kasar', 'Nonkasar']  # Only these top-level directories define labels.
VIDEO_EXTENSIONS = {'.mp4', '.webm', '.mkv', '.mov', '.avi', '.m4v', '.mpeg', '.mpg', '.3gp', '.ts'}
FFMPEG_EXE = shutil.which('ffmpeg') or r'C:\ffmpeg\bin\ffmpeg.exe'
FFMPEG_VERSION = None
FFMPEG_TIMEOUT = 600  # Decoder timeout per file; not an audio-duration limit.
SR = 22050
DURATION = None  # Decode the entire recording; never crop by duration.
SEED = 42
AUGMENT = True
SAMPLES_PER_CLASS = None  # Use every supported audio file in every class.
TEST_SIZE = 0.30
VAL_SIZE = 0.20  # Fraction of the remaining development/training partition.
RUN_CV = True
SVM_C = 1.0  # Prespecified baseline; do not tune using test results.
SVM_GAMMA = "scale"
OUTPUT_DIR = Path("reviewer_results_video_binary")
CACHE_DIR = Path(".audio_feature_cache_video")
CACHE_SCHEMA = "full-video-binary-v1"  # Increment after changing feature/augmentation definitions.

# Target ukuran tetap
N_MELS = 64
SPEC_TIME_FRAMES = 256  # Area-average the entire time axis; never take just the first frames.
TARGET_VECTOR_DIM = 183       # target panjang vektor fitur

K_FOLDS = 5                   # jumlah fold untuk cross-validation
EPOCHS_DNN = 40
EPOCHS_CNN = 40

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---------- Decode complete audio from a video's first audio stream ----------
def check_ffmpeg():
    flags = subprocess.CREATE_NO_WINDOW if os.name == 'nt' else 0
    try:
        result = subprocess.run([FFMPEG_EXE, '-version'], capture_output=True,
                                text=True, check=True, timeout=15, creationflags=flags)
    except (OSError, subprocess.SubprocessError) as exc:
        raise RuntimeError('FFmpeg is required. Install FFmpeg or set FFMPEG_EXE to ffmpeg.exe.') from exc
    return result.stdout.splitlines()[0]


def load_full_audio(media_path, sr=SR):
    # No -t/-ss: decode the entire first audio track; ignore video and image streams.
    command = [FFMPEG_EXE, '-nostdin', '-hide_banner', '-loglevel', 'error',
               '-i', str(Path(media_path).resolve()), '-map', '0:a:0', '-vn', '-sn', '-dn',
               '-ac', '1', '-ar', str(sr), '-acodec', 'pcm_f32le', '-f', 'f32le', 'pipe:1']
    flags = subprocess.CREATE_NO_WINDOW if os.name == 'nt' else 0
    try:
        result = subprocess.run(command, capture_output=True, timeout=FFMPEG_TIMEOUT,
                                check=False, creationflags=flags)
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(f'Audio decoding exceeded {FFMPEG_TIMEOUT}s: {media_path}') from exc
    if result.returncode != 0:
        message = result.stderr.decode('utf-8', errors='replace')[-2000:]
        raise ValueError(f'FFmpeg failed (missing audio stream or unreadable media): {message}')
    waveform = np.frombuffer(result.stdout, dtype='<f4').astype(np.float32, copy=True)
    if waveform.size == 0 or not np.isfinite(waveform).all():
        raise ValueError(f'Empty or nonfinite decoded audio: {media_path}')
    return waveform, sr


# ---------- Robust helpers (ganti fungsi lama) ----------
def flatten_numeric(x):
    """
    Mengembalikan list angka float dari input x.
    - Jika x scalar -> [float(x)]
    - Jika x array 1D/2D -> flatten -> list of floats
    - Jika x None atau kosong -> []
    """
    if x is None:
        return []
    arr = np.asarray(x)
    if arr.size == 0:
        return []
    arr = np.squeeze(arr)
    if np.isscalar(arr) or arr.ndim == 0:
        try:
            return [float(arr)]
        except Exception:
            try:
                return [float(np.asarray(arr).item())]
            except Exception:
                return []
    return arr.reshape(-1).astype(np.float32).tolist()

def safe_scalar(x, default=0.0):
    """
    Kembalikan float scalar jika x berisi satu elemen, else default or mean.
    """
    if x is None:
        return float(default)
    a = np.asarray(x)
    if a.size == 0:
        return float(default)
    if a.size == 1:
        try:
            return float(a.reshape(-1)[0])
        except Exception:
            return float(default)
    try:
        return float(a.mean())
    except Exception:
        return float(default)

def prepare_waveform(y):
    """Preserve every sample; only pad extremely short audio for spectral analysis."""
    y = np.asarray(y, dtype=np.float32)
    if y.ndim != 1 or y.size == 0 or not np.isfinite(y).all():
        raise ValueError("Audio must be a nonempty finite mono waveform")
    return np.pad(y, (0, max(0, 4096 - len(y))))


def resize_full_time(mel_db, target_frames):
    """Area averaging includes every input frame, including the recording's end.
    Short inputs are interpolated across the complete duration.
    """
    mel_db = np.asarray(mel_db, dtype=np.float32)
    frames = mel_db.shape[1]
    if target_frames < 1 or frames < 1:
        raise ValueError("Frame counts must be positive")
    if frames == target_frames:
        return mel_db.copy()
    if frames < target_frames:
        old = np.arange(frames)
        new = np.linspace(0, frames - 1, target_frames)
        return np.asarray([np.interp(new, old, row) for row in mel_db], dtype=np.float32)
    # Integral of piecewise-constant frame values at equally spaced boundaries.
    edges = np.linspace(0, frames, target_frames + 1)
    whole = np.floor(edges).astype(int)
    frac = edges - whole
    cumulative = np.concatenate([np.zeros((len(mel_db), 1)),
                                 np.cumsum(mel_db, axis=1, dtype=np.float64)], axis=1)
    integral = cumulative[:, whole] + mel_db[:, np.minimum(whole, frames - 1)] * frac
    return (np.diff(integral, axis=1) / np.diff(edges)).astype(np.float32)


def safe_mean(x, axis=None):
    """Wrapper np.mean yang selalu mengembalikan scalar atau array 1D yang aman."""
    if x is None:
        return np.array([])
    a = np.asarray(x)
    if a.size == 0:
        return np.array([])
    return np.mean(a, axis=axis)

def safe_std(x, axis=None):
    if x is None:
        return np.array([])
    a = np.asarray(x)
    if a.size == 0:
        return np.array([])
    return np.std(a, axis=axis)

# ---------- Audio augmentation helpers ----------
def augment_audio(y, sr, rng):
    # Yield one full-length variant at a time to avoid retaining five waveforms.
    yield (y + 0.005 * rng.standard_normal(len(y))).astype(np.float32)
    for steps in (2, -2):
        yield librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)
    for rate in (0.9, 1.1):
        # Keep the complete stretched result, including its tail.
        yield librosa.effects.time_stretch(y, rate=rate)


# ---------- Fixed feature extraction (vector) ----------
def extract_fixed_vector_features(audio_path, sr=SR, duration=DURATION, target_dim=TARGET_VECTOR_DIM):
    waveform, sr = load_full_audio(audio_path, sr=sr)
    return extract_vector_from_waveform(prepare_waveform(waveform), sr, target_dim)

# Versi dari waveform (untuk augmentasi)
def extract_vector_from_waveform(y, sr=SR, target_dim=TARGET_VECTOR_DIM):
    try:
        y = prepare_waveform(y)
        feats = []

        mfcc20 = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        mfcc13 = mfcc20[:13]  # Reuse the same transform.
        feats += flatten_numeric(safe_mean(mfcc13, axis=1))
        feats += flatten_numeric(safe_std(mfcc13, axis=1))
        feats += flatten_numeric(safe_mean(mfcc20, axis=1))
        feats += flatten_numeric(safe_std(mfcc20, axis=1))

        mfcc_delta = librosa.feature.delta(mfcc13, mode='nearest')
        mfcc_delta2 = librosa.feature.delta(mfcc13, order=2, mode='nearest')
        feats += flatten_numeric(safe_mean(mfcc_delta, axis=1))
        feats += flatten_numeric(safe_std(mfcc_delta, axis=1))
        feats += flatten_numeric(safe_mean(mfcc_delta2, axis=1))
        feats += flatten_numeric(safe_std(mfcc_delta2, axis=1))

        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        feats += [safe_scalar(safe_mean(spectral_centroid))]
        feats += [safe_scalar(safe_std(spectral_centroid))]
        feats += [safe_scalar(safe_mean(spectral_rolloff))]
        feats += [safe_scalar(safe_std(spectral_rolloff))]
        feats += [safe_scalar(safe_mean(spectral_bandwidth))]
        feats += [safe_scalar(safe_std(spectral_bandwidth))]

        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        feats += flatten_numeric(safe_mean(chroma, axis=1))
        feats += flatten_numeric(safe_std(chroma, axis=1))

        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        feats += flatten_numeric(safe_mean(contrast, axis=1))
        feats += flatten_numeric(safe_std(contrast, axis=1))

        try:
            tonnetz = librosa.feature.tonnetz(y=y, sr=sr)
            feats += flatten_numeric(safe_mean(tonnetz, axis=1))
            feats += flatten_numeric(safe_std(tonnetz, axis=1))
        except Exception:
            feats += [0.0] * 6
            feats += [0.0] * 6

        try:
            tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
            feats += [safe_scalar(tempo)]
        except Exception:
            feats += [0.0]

        try:
            zcr = librosa.feature.zero_crossing_rate(y)
            feats += flatten_numeric(safe_mean(zcr))
            feats += flatten_numeric(safe_std(zcr))
        except Exception:
            feats += [0.0, 0.0]

        try:
            rms = librosa.feature.rms(y=y)
            feats += flatten_numeric(safe_mean(rms))
            feats += flatten_numeric(safe_std(rms))
        except Exception:
            feats += [0.0, 0.0]

        try:
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            mel_db = librosa.power_to_db(mel)
            feats += flatten_numeric(safe_mean(mel_db))
            feats += flatten_numeric(safe_std(mel_db))
            feats += flatten_numeric(np.max(mel_db))
            feats += flatten_numeric(np.min(mel_db))
        except Exception:
            feats += [0.0, 0.0, 0.0, 0.0]

        features_array = np.array(feats, dtype=np.float32)
        features_array = np.nan_to_num(features_array)

        if len(features_array) < target_dim:
            features_array = np.pad(features_array, (0, target_dim - len(features_array)), mode='constant')
        elif len(features_array) > target_dim:
            features_array = features_array[:target_dim]

        return features_array
    except Exception as e:
        print(f"❌ Error extracting vector from waveform: {e}")
        traceback.print_exc()
        return None

# ---------- Full-recording spectrogram ----------
def extract_fixed_spectrogram(audio_path, sr=SR, duration=None, n_mels=N_MELS, target_frames=SPEC_TIME_FRAMES):
    waveform, sr = load_full_audio(audio_path, sr=sr)
    return extract_spectrogram_from_waveform(waveform, sr, n_mels, target_frames)


def extract_spectrogram_from_waveform(y, sr=SR, n_mels=N_MELS, target_frames=SPEC_TIME_FRAMES):
    y = prepare_waveform(y)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    if not np.isfinite(mel_db).all():
        raise ValueError("Nonfinite spectrogram")
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    return resize_full_time(mel_db, target_frames)


def cached_file_features(audio_path, augmented=False):
    """Cache deterministic per-file transforms, never scalers or split-dependent data."""
    global FFMPEG_VERSION
    if FFMPEG_VERSION is None:
        FFMPEG_VERSION = check_ffmpeg()
    path = Path(audio_path).resolve()
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    content_hash = digest.hexdigest()
    settings = [CACHE_SCHEMA, content_hash, SR, N_MELS, SPEC_TIME_FRAMES,
                TARGET_VECTOR_DIM, SEED, augmented, version('librosa'), version('numpy'),
                FFMPEG_VERSION]
    key = hashlib.sha256(json.dumps(settings).encode()).hexdigest()
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_path = CACHE_DIR / (key + '.npz')
    if cache_path.exists():
        try:
            with np.load(cache_path, allow_pickle=False) as data:
                return data['vectors'], data['specs'], float(data['duration']), content_hash
        except (ValueError, OSError, EOFError):
            print(f'Rebuilding unreadable feature cache: {cache_path.name}')
    waveform, sr = load_full_audio(path, sr=SR)
    duration = len(waveform) / sr
    waveform = prepare_waveform(waveform)
    rng = np.random.default_rng(np.random.SeedSequence([SEED, int(content_hash[:16], 16)]))
    variants = augment_audio(waveform, sr, rng) if augmented else (waveform,)
    vectors, specs = [], []
    for variant in variants:
        vec = extract_vector_from_waveform(variant, sr)
        spec = extract_spectrogram_from_waveform(variant, sr)
        if vec is None or vec.shape != (TARGET_VECTOR_DIM,) or not np.isfinite(vec).all():
            raise ValueError(f'Invalid vector features: {path}')
        vectors.append(vec); specs.append(spec)
    vectors, specs = np.asarray(vectors, dtype=np.float32), np.asarray(specs, dtype=np.float32)
    temporary = cache_path.with_suffix('.tmp.npz')
    np.savez_compressed(temporary, vectors=vectors, specs=specs, duration=duration)
    temporary.replace(cache_path)
    return vectors, specs, duration, content_hash


# ---------- Collect original recordings only ----------
def collect_deep_dataset(data_dir, samples_per_class=SAMPLES_PER_CLASS):
    root = Path(data_dir)
    if not root.is_dir():
        raise FileNotFoundError(f"Dataset directory not found: {root}")
    dirs = [root / name for name in CLASS_NAMES]
    missing = [str(d) for d in dirs if not d.is_dir()]
    if missing:
        raise FileNotFoundError(f'Missing class directories: {missing}')
    names = list(CLASS_NAMES)
    if len(names) < 2:
        raise ValueError("At least two class directories are required.")
    vectors, specs, labels, paths, skipped = [], [], [], [], []
    manifest = []
    inventory = []
    seen_hashes = {}
    rng = np.random.default_rng(SEED)
    for label, directory in enumerate(dirs):
        files = sorted(f for f in directory.rglob("*") if f.is_file()
                       and ".ipynb_checkpoints" not in f.relative_to(directory).parts
                       and f.suffix.lower() in
                       VIDEO_EXTENSIONS)
        inventory.append({'class': names[label], 'discovered_videos': len(files)})
        print(f'{names[label]}: {len(files)} video files discovered')
        if samples_per_class is not None and len(files) > samples_per_class:
            files = sorted(rng.choice(files, size=samples_per_class, replace=False))
        iterator = tqdm(files, desc=directory.name, unit='file') if TQDM_AVAILABLE else files
        for path in iterator:
            try:
                vectors_file, specs_file, duration, content_hash = cached_file_features(path)
                vec, spec = vectors_file[0], specs_file[0]
                if vec is None or spec is None:
                    raise ValueError("Feature extraction failed")
                if content_hash in seen_hashes:
                    raise ValueError(f"Duplicate file content; already included: {seen_hashes[content_hash]}")
                seen_hashes[content_hash] = str(path)
                vectors.append(vec); specs.append(spec); labels.append(label)
                manifest.append({'path': str(path.resolve()), 'class': names[label],
                                 'duration_seconds': duration, 'sha256': content_hash})
                paths.append(str(path.resolve()))
            except Exception as exc:
                skipped.append({'path': str(path), 'error': str(exc)})
                print(f"Skipped {path}: {exc}")
    write_json('skipped_files.json', skipped)
    write_csv('video_inventory.csv', inventory)
    write_csv('dataset_manifest.csv', manifest)
    print(f'Included {len(manifest)} originals; skipped {len(skipped)} files (see log).')
    counts = np.bincount(labels, minlength=len(names))
    print('Original recordings per class:', dict(zip(names, counts.tolist())))
    if np.any(counts < 5):
        raise ValueError("Too few original recordings in a class for stratified train/validation/test splits.")
    return (np.asarray(vectors, dtype=np.float32), np.asarray(specs, dtype=np.float32),
            np.asarray(labels, dtype=np.int64), names, paths)


def training_data(indices, X_vec, X_spec, y, paths):
    # Augmented copies can only originate from this training partition.
    vectors, specs, labels = list(X_vec[indices]), list(X_spec[indices]), list(y[indices])
    if AUGMENT:
        iterator = tqdm(indices, desc='Training augmentation', unit='file') if TQDM_AVAILABLE else indices
        for idx in iterator:
            augmented_vec, augmented_spec, _, _ = cached_file_features(paths[idx], augmented=True)
            vectors.extend(augmented_vec); specs.extend(augmented_spec)
            labels.extend([y[idx]] * len(augmented_vec))
    return (np.asarray(vectors, dtype=np.float32),
            np.asarray(specs, dtype=np.float32)[..., np.newaxis], np.asarray(labels, dtype=np.int64))

# ---------- Build DNN for vector features ----------
def build_dnn(input_dim, num_classes):
    model = Sequential([
        Dense(512, input_dim=input_dim),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.35),

        Dense(256),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.35),

        Dense(128),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.35),

        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# ---------- Build CNN for spectrograms ----------
def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv2D(32, (3,3), padding='same', input_shape=input_shape),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((2,2)),

        Conv2D(64, (3,3), padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((2,2)),

        Conv2D(128, (3,3), padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((2,2)),

        GlobalAveragePooling2D(),
        Dense(128),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.5),

        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# ---------- Reproducible comparison and exported evidence ----------
METRICS = ('accuracy', 'precision_macro', 'recall_macro', 'f1_macro')


def write_json(name, data):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / name).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')


def write_csv(name, rows):
    if not rows:
        return
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with (OUTPUT_DIR / name).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def split_validation(indices, y, seed):
    return train_test_split(indices, test_size=VAL_SIZE, random_state=seed, stratify=y[indices])


def metric_row(y_true, y_pred, names):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(names)), average='macro', zero_division=0)
    return dict(accuracy=float(accuracy_score(y_true, y_pred)),
                precision_macro=float(precision), recall_macro=float(recall), f1_macro=float(f1))


def evaluate_predictions(tag, name, y_true, y_pred, names, paths):
    row = {'experiment': tag, 'model': name, **metric_row(y_true, y_pred, names)}
    labels = np.arange(len(names))
    report = classification_report(y_true, y_pred, labels=labels, target_names=names,
                                   output_dict=True, zero_division=0)
    write_json(f'{tag}_{name}_report.json', report)
    write_csv(f'{tag}_{name}_predictions.csv', [
        {'path': path, 'true_class': names[int(actual)], 'predicted_class': names[int(pred)]}
        for path, actual, pred in zip(paths, y_true, y_pred)])
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    write_json(f'{tag}_{name}_confusion_matrix.json', {'class_names': names, 'matrix': cm.tolist()})
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=names, yticklabels=names, ax=ax)
    ax.set(title=f'{tag}: {name}', xlabel='Predicted', ylabel='Actual')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'{tag}_{name}_confusion_matrix.png', dpi=180)
    plt.close(fig)
    print(row)
    return row


def run_comparison(tag, train_idx, val_idx, test_idx, X_vec, X_spec, y, names, paths):
    assert set(train_idx).isdisjoint(val_idx)
    assert set(train_idx).isdisjoint(test_idx)
    assert set(val_idx).isdisjoint(test_idx)
    tf.keras.utils.set_random_seed(SEED)
    write_csv(f'{tag}_split.csv', [
        {'index': int(i), 'path': paths[i], 'class': names[int(y[i])], 'partition': partition}
        for partition, indices in [('train', train_idx), ('validation', val_idx), ('test', test_idx)]
        for i in indices])
    train_vec, train_spec, train_y = training_data(train_idx, X_vec, X_spec, y, paths)
    assert train_vec.shape[1] == TARGET_VECTOR_DIM
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_vec)
    val_scaled, test_scaled = scaler.transform(X_vec[val_idx]), scaler.transform(X_vec[test_idx])
    write_json(f'{tag}_sample_counts.json', {
        'training_original': len(train_idx), 'training_including_augmentation': len(train_y),
        'validation_original': len(val_idx), 'test_original': len(test_idx)})
    rows = []
    svm = SVC(kernel='rbf', C=SVM_C, gamma=SVM_GAMMA)
    svm.fit(train_scaled, train_y)
    rows.append(evaluate_predictions(tag, 'SVM_RBF', y[test_idx], svm.predict(test_scaled),
                                     names, [paths[i] for i in test_idx]))
    if tag == 'holdout':
        joblib.dump({'model': svm, 'scaler': scaler, 'class_names': names}, OUTPUT_DIR / 'svm_rbf.pkl')
        joblib.dump({'scaler': scaler, 'class_names': names}, OUTPUT_DIR / 'audio_scaler_and_labels.pkl')
    for name, builder, train_x, val_x, test_x, epochs, batch in [
        ('DNN', lambda: build_dnn(TARGET_VECTOR_DIM, len(names)), train_scaled,
         val_scaled, test_scaled, EPOCHS_DNN, 64),
        ('CNN', lambda: build_cnn(train_spec.shape[1:], len(names)), train_spec,
         X_spec[val_idx, ..., np.newaxis], X_spec[test_idx, ..., np.newaxis], EPOCHS_CNN, 32)]:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)
        model = builder()
        history = model.fit(train_x, train_y, validation_data=(val_x, y[val_idx]),
                            epochs=epochs, batch_size=batch, verbose=1, callbacks=[
            EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4)])
        write_json(f'{tag}_{name}_history.json',
                   {key: [float(v) for v in values] for key, values in history.history.items()})
        predictions = np.argmax(model.predict(test_x, verbose=0), axis=1)
        rows.append(evaluate_predictions(tag, name, y[test_idx], predictions, names,
                                         [paths[i] for i in test_idx]))
        if tag == 'holdout':
            model.save(OUTPUT_DIR / f'{name.lower()}_model.keras')
        del model
    return rows


def main():
    global FFMPEG_VERSION
    FFMPEG_VERSION = check_ffmpeg()
    print(FFMPEG_VERSION)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    X_vec, X_spec, y, names, paths = collect_deep_dataset(DATA_DIR)
    # Reserve the common holdout before CV. It never participates in CV or early stopping.
    development_idx, test_idx = train_test_split(
        np.arange(len(y)), test_size=TEST_SIZE, random_state=SEED, stratify=y)
    train_idx, val_idx = split_validation(development_idx, y, SEED)
    write_json('experiment_config.json', {
        'seed': SEED, 'data_dir': str(DATA_DIR), 'class_names': names,
        'samples_per_class_cap': SAMPLES_PER_CLASS, 'sr': SR, 'duration_seconds': DURATION,
        'audio_scope': 'complete_first_audio_stream_of_video',
        'ffmpeg_version': FFMPEG_VERSION, 'video_extensions': sorted(VIDEO_EXTENSIONS),
        'label_definition': {'Kasar': 0, 'Nonkasar': 1}, 'cnn_time_resize': 'area_average_or_interpolate',
        'cache_schema': CACHE_SCHEMA, 'cnn_pooling': 'global_average',
        'vector_dim': TARGET_VECTOR_DIM, 'n_mels': N_MELS, 'spec_frames': SPEC_TIME_FRAMES,
        'augment_train_only': AUGMENT, 'test_size': TEST_SIZE, 'validation_fraction': VAL_SIZE,
        'cv_folds': K_FOLDS if RUN_CV else 0, 'svm_C': SVM_C, 'svm_gamma': SVM_GAMMA,
        'epochs_dnn': EPOCHS_DNN, 'epochs_cnn': EPOCHS_CNN,
        'precision_recall_averaging': 'macro', 'python': platform.python_version(),
        'packages': {pkg: version(pkg) for pkg in ['numpy', 'scikit-learn', 'librosa', 'tensorflow']}})
    cv_rows = []
    if RUN_CV:
        if np.min(np.bincount(y[development_idx], minlength=len(names))) < K_FOLDS:
            raise ValueError('Not enough original development recordings per class for K_FOLDS.')
        folds = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
        for fold, (fit_pos, eval_pos) in enumerate(folds.split(development_idx, y[development_idx]), 1):
            fit_idx, validation_idx = split_validation(development_idx[fit_pos], y, SEED + fold)
            cv_rows.extend(run_comparison(f'cv_{fold}', fit_idx, validation_idx,
                development_idx[eval_pos], X_vec, X_spec, y, names, paths))
            write_csv('cv_metrics.csv', cv_rows)
        summary = []
        for name in ('SVM_RBF', 'DNN', 'CNN'):
            row = {'model': name}
            for metric in METRICS:
                values = [r[metric] for r in cv_rows if r['model'] == name]
                row[metric + '_mean'] = float(np.mean(values))
                row[metric + '_std'] = float(np.std(values, ddof=1))
            summary.append(row)
        write_csv('cv_summary.csv', summary)
        print('CV mean and sample standard deviation:', summary)
    holdout_rows = run_comparison('holdout', train_idx, val_idx, test_idx,
                                  X_vec, X_spec, y, names, paths)
    write_csv('holdout_metrics.csv', holdout_rows)
    baseline = holdout_rows[0]
    write_csv('holdout_delta_vs_svm.csv', [
        {'model': row['model'], **{metric + '_delta': row[metric] - baseline[metric] for metric in METRICS}}
        for row in holdout_rows[1:]])
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(holdout_rows))
    for offset, metric in enumerate(METRICS):
        ax.bar(x + (offset - 1.5) * 0.2, [r[metric] for r in holdout_rows], 0.2, label=metric)
    ax.set_xticks(x, [r['model'] for r in holdout_rows])
    ax.set(ylim=(0, 1.05), ylabel='Score', title='Common held-out original recordings')
    ax.legend(); fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'holdout_comparison.png', dpi=200)
    plt.show()
    print(f'Experiment artifacts: {OUTPUT_DIR.resolve()}')
    return holdout_rows, cv_rows


In [ ]:
holdout_results, cv_results = main()


ffmpeg version 2024-08-28-git-b730defd52-full_build-www.gyan.dev Copyright (c) 2000-2024 the FFmpeg developers
Kasar: 922 video files discovered


Kasar:   0%|                                                              | 0/922 [00:00<?, ?file/s]D:\web\cnn_dnn_rnn_gan\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Kasar:  14%|███████                                             | 126/922 [01:14<07:25,  1.79file/s]

Skipped D:\web\cnn_clasification\archive\Dataset\Kasar\Babi\anjing--babi_lu------__TubeTrendy__7554772107004071175.mp4: Duplicate file content; already included: D:\web\cnn_clasification\archive\Dataset\Kasar\Anjing\anjing--babi_lu------__TubeTrendy__7554772107004071175.mp4


Kasar:  35%|██████████████████▍                                 | 327/922 [03:18<05:53,  1.68file/s]

Skipped D:\web\cnn_clasification\archive\Dataset\Kasar\Bangsat\_CapCut______TubeTrendy__7371810231837412614.mp4: Duplicate file content; already included: D:\web\cnn_clasification\archive\Dataset\Kasar\Anjing\_CapCut______TubeTrendy__7371810231837412614.mp4


Kasar:  41%|█████████████████████▌                              | 382/922 [03:46<04:25,  2.04file/s]

... (24 outputs left out)

Skipped D:\web\cnn_clasification\archive\Dataset\Kasar\Tolol\upsss___ojingtogether__ojingjepara__inpoorkesjepara__masukbranda__viral__fyp___TubeTrendy__7536940002891042053.mp4: Duplicate file content; already included: D:\web\cnn_clasification\archive\Dataset\Kasar\Anjing\upsss___ojingtogether__ojingjepara__inpoorkesjepara__masukbranda__viral__fyp___TubeTrendy__7536940002891042053.mp4


Kasar: 100%|████████████████████████████████████████████████████| 922/922 [09:07<00:00,  1.68file/s]


Nonkasar: 768 video files discovered


Nonkasar:  52%|█████████████████████████▍                       | 399/768 [22:43<17:49,  2.90s/file]

## Pelaporan untuk reviewer

- **Reviewer A:** laporkan SVM-RBF (C=1, gamma=scale) versus DNN pada fitur 183 dimensi identik, serta CNN pada spectrogram, dengan pembagian video dan metrik yang sama. Gunakan `holdout_metrics.csv`, `cv_summary.csv`, dan `holdout_delta_vs_svm.csv`. SVM tidak dituning menggunakan test; early stopping DNN/CNN hanya memakai validasi internal.
- **Reviewer B:** jelaskan gap melalui studi yang benar-benar dibaca dan disitasi. Kontribusi yang dapat diuji adalah perbandingan baseline konvensional dan deep learning untuk klasifikasi audio video berlabel Kasar/Nonkasar dengan protokol bersama. Jangan mengklaim kebaruan atau keunggulan sebelum literatur dan hasil mendukungnya.
- Laporkan distribusi dua kelas setelah kegagalan ekstraksi/duplikat, prosedur anotasi, asal video, dan durasi. Nama folder dipakai sebagai label yang diberikan pengguna, bukan verifikasi otomatis bahwa isinya kasar atau nonkasar. Periksa konsistensi anotasi dan potensi tumpang tindih sumber video/penutur.
- Bahas kesalahan per kelas dan selisih macro F1 terhadap SVM, termasuk bila SVM lebih baik. Selisih skor dan simpangan baku fold bukan bukti signifikansi statistik. Perbedaan DNN/CNN mencakup representasi input dan arsitektur.
- Semua angka harus dihitung ulang untuk dataset biner ini. Jangan menggabungkan hasil sebelumnya dari delapan kelas kata atau audio dua detik. Notebook ini hanya memakai audio; tidak mengevaluasi informasi visual video.
